In [4]:
# =========================================================
# CLASSIFICATION MODELING
# =========================================================

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# -----------------------------
# 1. LOAD DATA
# -----------------------------
DATA_PATH = r"../data/processed/emi_featured_final.csv"
df = pd.read_csv(DATA_PATH)

print("Original dataset shape:", df.shape)

# -----------------------------
# 2. USE SAMPLE FIRST (FAST TEST)
# -----------------------------
df = df.sample(50000, random_state=42)

print("Sample dataset shape:", df.shape)

# =========================================================
# STEP 3: DEFINE FEATURES AND TARGET 
# =========================================================

target_col = "emi_eligibility"

drop_cols = [
    "emi_eligibility",
    "max_monthly_emi",
    "emi_capacity_gap",
    "requested_emi_estimate",
    "requested_emi_to_income_ratio"
]

X = df.drop(columns=drop_cols)
y = df[target_col]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nColumns used for modeling:")
print(X.columns.tolist())

# -----------------------------
# 4. TRAIN-TEST SPLIT
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTrain shape:", X_train.shape)
print("Test shape :", X_test.shape)

# -----------------------------
# 5. COLUMN TYPES
# -----------------------------
categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

print("\nCategorical columns:", len(categorical_cols))
print("Numerical columns :", len(numerical_cols))

# -----------------------------
# 6. PREPROCESSOR
# -----------------------------
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

# -----------------------------
# 7. LIGHTER MODELS
# -----------------------------
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=500,
        class_weight="balanced",
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=80,
        max_depth=12,
        min_samples_split=10,
        min_samples_leaf=4,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=80,
        learning_rate=0.1,
        random_state=42
    )
}

# -----------------------------
# 8. TRAIN + EVALUATE
# -----------------------------
results = []
trained_pipelines = {}

for model_name, model in models.items():
    print(f"\nTraining: {model_name}")

    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

    roc_auc = np.nan
    if hasattr(pipeline, "predict_proba"):
        y_prob = pipeline.predict_proba(X_test)
        roc_auc = roc_auc_score(y_test, y_prob, multi_class="ovr", average="weighted")

    results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC-AUC": roc_auc
    })

    trained_pipelines[model_name] = pipeline

# -----------------------------
# 9. RESULTS TABLE
# -----------------------------
results_df = pd.DataFrame(results).sort_values(by="F1 Score", ascending=False).reset_index(drop=True)

for col in ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"]:
    results_df[col] = results_df[col].round(4)

print("\nClassification Results:")
print(results_df)

Original dataset shape: (404800, 49)
Sample dataset shape: (50000, 49)
X shape: (50000, 44)
y shape: (50000,)

Columns used for modeling:
['age', 'gender', 'marital_status', 'education', 'monthly_salary', 'employment_type', 'years_of_employment', 'company_type', 'house_type', 'monthly_rent', 'family_size', 'dependents', 'school_fees', 'college_fees', 'travel_expenses', 'groceries_utilities', 'other_monthly_expenses', 'existing_loans', 'current_emi_amount', 'credit_score', 'bank_balance', 'emergency_fund', 'emi_scenario', 'requested_amount', 'requested_tenure', 'total_expenses', 'disposable_income', 'expense_to_income_ratio', 'emi_to_income_ratio', 'savings_to_income_ratio', 'loan_to_income_ratio', 'emergency_fund_months', 'dependents_ratio', 'income_per_year_exp', 'disposable_income_ratio', 'high_financial_stress', 'low_emergency_fund_flag', 'high_loan_burden_flag', 'credit_score_bucket', 'experience_bucket', 'salary_bucket', 'salary_credit_interaction', 'disposable_credit_interaction'

Three classification models were trained to predict EMI eligibility: Logistic Regression, Random Forest Classifier, and Gradient Boosting Classifier.

After removing data leakage features, Gradient Boosting achieved the best performance with an accuracy of 94.39% and F1 score of 0.9244. This indicates that the relationship between financial variables and EMI eligibility is non-linear and better captured by ensemble models.

Random Forest also performed well, while Logistic Regression showed comparatively lower performance due to its linear nature.

The ROC-AUC scores indicate strong class separability, and Gradient Boosting was selected as the final classification model for EMI eligibility prediction.

    'Care was taken to remove features that could introduce data leakage, such as EMI capacity and derived approval-related variables, ensuring that the model learns from realistic input features'.

In [6]:
# =========================================================
# STEP 10: MLFLOW LOGGING - CLASSIFICATION
# =========================================================

import mlflow
import mlflow.sklearn

mlflow.set_experiment("EMI_Predict_AI_Classification")

classification_results_df = results_df.copy()

for _, row in classification_results_df.iterrows():
    model_name = row["Model"]
    model = trained_pipelines[model_name]

    with mlflow.start_run(run_name=f"clf_{model_name}"):
        mlflow.log_param("problem_type", "classification")
        mlflow.log_param("model_type", model_name)

        mlflow.log_metric("accuracy", float(row["Accuracy"]))
        mlflow.log_metric("precision", float(row["Precision"]))
        mlflow.log_metric("recall", float(row["Recall"]))
        mlflow.log_metric("f1_score", float(row["F1 Score"]))
        mlflow.log_metric("roc_auc", float(row["ROC-AUC"]))

        mlflow.sklearn.log_model(model, name="model")

print("Classification models logged to MLflow successfully.")

2026/04/08 19:10:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/08 19:10:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/08 19:10:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mec

Classification models logged to MLflow successfully.


Final Conclusion:
Among the classification models tested, Gradient Boosting Classifier achieved the best performance after removing leakage-prone features. It was selected as the final model for EMI eligibility prediction because it provided the highest F1-score and accuracy while handling non-linear financial patterns effectively.

MLflow was used to track classification and regression experiments separately. It logged model metrics, parameters, and model artifacts, enabling systematic comparison and final model selection in an organized and reproducible way.

In [ ]:
# =========================================================
# SAVE FINAL CLASSIFICATION MODEL
# =========================================================

import os
import joblib

# Create models folder (relative path)
os.makedirs("models", exist_ok=True)

best_classification_model = trained_pipelines["Gradient Boosting"]

joblib.dump(
    best_classification_model,
    "models/best_classification_model.pkl"
)

print("Best classification model saved successfully.")

Best classification model saved successfully.


In [8]:
# -----------------------------
# COMPARE ORIGINAL vs PREDICTED
# -----------------------------

# Select best model (change if needed)
best_model = trained_pipelines["Gradient Boosting"]

# Predict on full dataset
df["predicted_eligibility"] = best_model.predict(X)

# BEFORE (Original)
print("🔹 Original Distribution:")
print(df["emi_eligibility"].value_counts(normalize=True))

# AFTER (Predicted)
print("\n🔹 Predicted Distribution:")
print(df["predicted_eligibility"].value_counts(normalize=True))

🔹 Original Distribution:
emi_eligibility
Not_Eligible    0.77422
Eligible        0.18384
High_Risk       0.04194
Name: proportion, dtype: float64

🔹 Predicted Distribution:
predicted_eligibility
Not_Eligible    0.81000
Eligible        0.18906
High_Risk       0.00094
Name: proportion, dtype: float64


The model improves decision clarity by minimizing the ambiguous ‘High Risk’ category. Instead, it leverages financial indicators to make more actionable binary decisions, which is more aligned with how lending systems operate in practice.